In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())
exec(open(os.path.join(path_git, 'Data', 'BLS', 'config', 'BLS Functions.py')).read())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

***

Monthly job growth

***

In [ ]:
## Importing ---
indicator_name = 'Jobs_1'
plot_name = 'monthly_line_test'
export = False

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)



## Organizing ---

df_plot = df_jobs.copy()

# Growth rate
df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])

# SACOG roll up
conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])
df_plot['Jobs_GR'] = round(df_plot['Jobs_GR'], 2)
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})


display(df_plot.head())


## Plotting ---

color_map = {
     "SACOG":"#9DC209",
     "National": "#1F45FC",
     "Peer MSA": "#1E90FF"
}

fig = px.line(df_plot, x='date_', y='Growth Rate', color='Groups', color_discrete_map=color_map)

title = 'Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas'
fig.update_layout(xaxis_title = 'Date')
fig.update_yaxes(tick0=0, dtick=2, ticksuffix='%')
fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")


plot_agol(export=export)


***

Bar chart and line plot together

***

In [ ]:
## Importing ---
indicator_name = 'Jobs_1'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Oragnizing ---

df_plot = df_jobs.copy()
month = '09'

df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
df_plot = df_plot.reset_index(drop = True)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])

conditions = [   
         df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2000, 2007, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2008, 2011, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2012, 2019, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2020, 2020, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2021, 2024, 1)))
             ]
choices = ["Pre Recession<br>(2000-2008)", "Recession<br>(2008-2011)", "Post Recession<br>(2011-2020)", "Covid<br>(2020)", "Post Covid<br>(2020-2024)"]
df_plot["Period"] = np.select(conditions, choices)

df_plot = df_plot.dropna()
df_plot1 = df_plot.groupby(['Groups', 'Period'], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2 = df_plot.groupby(['Groups'          ], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2['Period'] = 'Total<br>(2000-2024)'
df_plot = pd.concat([df_plot1, df_plot2])
categories = ["Total<br>(2000-2024)", "Pre Recession<br>(2000-2008)", "Recession<br>(2008-2011)", "Post Recession<br>(2011-2020)", "Covid<br>(2020)", "Post Covid<br>(2020-2024)"]
df_plot['Period_Sort'] = pd.Categorical(df_plot['Period'], categories)
df_plot = df_plot.sort_values(by = ['Groups', 'Period_Sort'], ascending = [True, True])
df_plot = df_plot.drop(['Period_Sort'], axis = 1)
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate', 'Period':'Time Period'})

display(df_plot.head())


## Plotting ---

color_map = {
     "SACOG":"#9DC209",
     "National": "#1F45FC",
     "Peer MSA": "#1E90FF"
}

df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)
# df_plot = df_plot.rename(columns = {'Group':'Geography'})
fig1 = px.bar(df_plot, x='Time Period', y='Growth Rate'
             , color='Groups'
             , color_discrete_map=color_map
             , barmode='group'
             , hover_name = 'Time Period')
fig1.update_yaxes(tick0=0, dtick=2, ticksuffix='%')
fig1.update_xaxes(tickangle=0)
fig1.update_layout(xaxis_title=None, xaxis=dict(tickfont = dict(size=11)))

title = 'Annual Job Growth Comparison: Sacramento, National, and other Mid-Sized Metro Areas (September)'
fig1.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig1.update_traces(hovertemplate="Growth Rate: %{y}")
fig1.show()


## Importing ---
indicator_name = 'Jobs_1'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Organizing ---

df_plot = df_jobs.copy()
month = '09'

df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
df_plot = df_plot.reset_index(drop = True)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])


df_plot = df_plot.dropna()
df_plot = df_plot.sort_values(by = ['Groups', 'date_'], ascending = [True, True])
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})

df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)
df_plot = df_plot.rename(columns = {'Groups':'Geography'})

display(df_plot.head())


## Plotting ---

color_map = {
     "SACOG":"#9DC209",
     "National": "#1F45FC",
     "Peer MSA": "#1E90FF"
}

fig2 = px.line(df_plot, x = 'date_', y = 'Growth Rate', color = 'Geography', color_discrete_map=color_map)
fig2.update_yaxes(tick0=0, dtick=1)
fig2.add_vline(x = '2009-01-01', line_dash = 'dash')
fig2.add_vline(x = '2012-01-01', line_dash = 'dash')
fig2.add_vline(x = '2020-01-01', line_dash = 'dash')
fig2.add_vline(x = '2021-03-01', line_dash = 'dash')
fig2.add_annotation(x='2005-01-01', y = -6.5, text="Pre-Recession" , showarrow= False)
fig2.add_annotation(x='2010-07-01', y = -6.5, text="Recession"     , showarrow= False)
fig2.add_annotation(x='2016-01-01', y = -6.5, text="Post-Recession", showarrow= False)
fig2.add_annotation(x='2020-07-20', y = -6.5, text="Covid"         , showarrow= False)
fig2.add_annotation(x='2022-09-01', y = -6.5, text="Post-Covid"    , showarrow= False)
fig2.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')
fig2.update_xaxes(dtick="M24", tickformat="%Y", ticklabelmode="period")
fig2.update_layout(xaxis_title  = 'Year')


title = 'Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas (September)'
fig2.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig2.update_traces(hovertemplate="Year: %{x}<br>Growth Rate: %{y}")


fig2.show()

In [ ]:
# https://stackoverflow.com/questions/56727843/how-can-i-create-subplots-with-plotly-express


## Plotting ---

indicator_name = 'Jobs_1'
plot_name = 'side_by_side_bar_line_test'
export = False


fig1_traces = []
fig2_traces = []


for trace in range(len(fig1["data"])):
    fig1_traces.append(fig1["data"][trace])

for trace in range(len(fig2["data"])):
    fig2["data"][trace]['showlegend'] = False
    fig2_traces.append(fig2["data"][trace])

# Create a 1x2 subplot
fig = sp.make_subplots(rows = 1, cols = 2
                       , subplot_titles=('<span style="font-size: 13px;">By Time Period (September to September)</span>', 
                                         '<span style="font-size: 13px;">By Year (September to September)</span>')
                      )

# Get the Express fig broken down as traces and add the traces to the proper plot within the subplot
for traces in fig1_traces:
    fig.append_trace(traces, row = 1, col = 1)
for traces in fig2_traces:
    fig.append_trace(traces, row = 1, col = 2)

# fig.update_layout(legend_title=None, title='Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas (by Presidential Administration) (November)')
fig.add_vline(x = '2009-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2012-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2020-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2021-03-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_annotation(x='2005-01-01', y = -7, text='<span style="font-size: 7px;">Pre-Recession</span>' , showarrow= False, row=1, col=2)
fig.add_annotation(x='2010-07-01', y = -7, text='<span style="font-size: 7px;">Recession</span>'     , showarrow= False, row=1, col=2)
fig.add_annotation(x='2016-01-01', y = -7, text='<span style="font-size: 7px;">Post-Recession</span>', showarrow= False, row=1, col=2)
fig.add_annotation(x='2020-07-20', y = -7, text='<span style="font-size: 7px;">Covid</span>'         , showarrow= False, row=1, col=2)
fig.add_annotation(x='2022-07-01', y = -7, text='<span style="font-size: 7px;">Post-Covid</span>'    , showarrow= False, row=1, col=2)
fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray', row=1, col=2)
            

title='Job Growth Comparison: Sacramento, National, and other Mid-Sized Metro Areas'
fig.update_yaxes(tick0=0, dtick=2, ticksuffix='%', range = [-7,7])
fig.update_xaxes(tickangle=0)
fig.update_xaxes(showticklabels=False, showgrid=False, row=1, col=2)

# fig.show()
plot_agol(export=export)

***

Seasonal decomposition

***

In [ ]:
## Importing ---
indicator_name = 'Jobs_1'
plot_name = 'monthly_line_test'
export = False

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)



## Organizing ---

df_plot = df_jobs.copy()

# Growth rate
df_plot = df_plot[df_plot['Sector'] != 'All']
df_plot = df_plot.sort_values(['Geography', 'Sector', 'date_'], ascending = [True, True, True])

df_plot = df_plot.groupby(['date_', 'Geography', 'Sector'], as_index = False).agg(Value = ('Value', 'sum'))
df_plot = df_plot.sort_values(['Geography', 'date_', 'Sector'], ascending = [True, True, True])
df_plot = df_plot[df_plot['Geography'] == 'Sacramento--Roseville--Arden-Arcade, CA']
df_plot = df_plot[df_plot['Sector'] == 'Government']
df_plot['moving_avg'] = df_plot['Value'].rolling(window=12).mean()


display(df_plot.head())


# ## Plotting ---


fig = px.line(df_plot, x='date_', y='Value', markers = False, color = 'Sector')
fig['data'][0]['line']['color']='#1E90FF'

fig.add_trace(go.Scatter(x=df_plot["date_"], y=df_plot['moving_avg']
                         , name = 'Seasonal Decomposition'
                         , line=go.scatter.Line(color="gray", dash="dot")
                        ))

title = 'Monthly Job Growth: Sacramento Goverment'
fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")


plot_agol(export=export)

***

Line plots by sector

***

In [ ]:
## Importing ---
indicator_name = 'Jobs_2'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Organizing ---

for sector in df_jobs['Sector'].unique():
    
    df_plot = df_jobs.copy()
    month = '09'
    
    df_plot = df_plot[df_plot['Sector'] == sector]
    df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
    df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
    df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
    df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
    df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
    df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
    df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
    df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
    df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
    df_plot = df_plot.reset_index(drop = True)
    
    wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average
    
    conditions = [   
           df_plot['Geography'].str.contains('Sac|Yuba')
        , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
        ,  df_plot['Geography'].str.contains('National')
                 ]
    choices = ['SACOG', 'Peer MSA', 'National']
    df_plot['Groups'] = np.select(conditions, choices)
    df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
    df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])
    
    
    df_plot = df_plot.dropna()
    df_plot = df_plot.sort_values(by = ['Groups', 'date_'], ascending = [True, True])
    df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})
    
    df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)
    df_plot = df_plot.rename(columns = {'Groups':'Geography'})
    
    display(df_plot.head())
    
    
    ## Plotting ---
    
    color_map = {
         "SACOG":"#9DC209",
         "National": "#1F45FC",
         "Peer MSA": "#1E90FF"
    }
    
    fig = px.line(df_plot, x = 'date_', y = 'Growth Rate', color = 'Geography', color_discrete_map=color_map)
    fig.update_yaxes(tick0=0, dtick=1)
    fig.add_vline(x = '2009-01-01', line_dash = 'dash')
    fig.add_vline(x = '2012-01-01', line_dash = 'dash')
    fig.add_vline(x = '2020-01-01', line_dash = 'dash')
    fig.add_vline(x = '2021-03-01', line_dash = 'dash')
    fig.add_annotation(x='2005-01-01', y = -6.5, text="Pre-Recession" , showarrow= False)
    fig.add_annotation(x='2010-07-01', y = -6.5, text="Recession"     , showarrow= False)
    fig.add_annotation(x='2016-01-01', y = -6.5, text="Post-Recession", showarrow= False)
    fig.add_annotation(x='2020-07-20', y = -6.5, text="Covid"         , showarrow= False)
    fig.add_annotation(x='2022-09-01', y = -6.5, text="Post-Covid"    , showarrow= False)
    fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')
    fig.update_xaxes(dtick="M24", tickformat="%Y", ticklabelmode="period")
    fig.update_layout(xaxis_title  = 'Year')
    
    
    title = f'Monthly {sector} Job Growth: Sacramento and other Mid-Sized Metro Areas (September)'
    fig.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
    fig.update_traces(hovertemplate="Year: %{x}<br>Growth Rate: %{y}")

    fig.show()